# 算法过程

In [1]:
import os, sys
from pathlib import Path

root = Path.cwd().resolve()
while root != root.parent and not (root / "timing").is_dir(): root = root.parent
if not (root / "timing").is_dir(): raise RuntimeError(f"cannot find project root from cwd={Path.cwd()}")
if str(root) not in sys.path: sys.path.insert(0, str(root))
print(f"project_root={root}")

project_root=/Users/akulaku/3sy11


In [27]:
import duckdb, pandas as pd, plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
from timing.analysis.algo.retracement.algo import compute_retracement

# ═══════════════════════════════════════════════════
# ★ 配置区：修改这里选择标的和调参 ★
# ═══════════════════════════════════════════════════
SYMBOL   = "515300.OF"      # 要验证的标的
INTERVAL = "1d"             # K线周期
DATA_DB  = str(root / "timing" / "cache" / "data.duckdb")

# ── swing 拐点参数 ──
PIVOT_WINDOWS       = [(5, 5), (8, 8)]      # (left_bars, right_bars) 组合，越大拐点越稀疏
ZIGZAG_THRESHOLDS   = [0.05, 0.10]          # zigzag 最小涨跌幅阈值（5%、10%）
REGRESSION_WINDOWS  = [50, 100]             # 回归偏差窗口大小
WEIGHTS = {                                  # 各标注方法权重（用于置信度加权合并）
    'pivot_5': 0.5, 'pivot_8': 1.0,
    'zigzag_5': 0.5, 'zigzag_10': 1.0,
    'reg_50': 0.5, 'reg_100': 1.0,
}
CLUSTER_TOLERANCE_PCT = 0.005               # 聚类容差（价格 ±0.5% 内合并为同一水平线）
MIN_CLUSTER_CONF      = 0.3                 # 置信度阈值（低于此的拐点不参与聚类）

# ── fib 回撤参数 ──
MIN_LEG_SPAN_PCT = 0.03                     # 趋势腿最小幅度（span < 3% 的腿被过滤）
MAX_RATIO_ERROR  = 0.05                     # fib 比率容差（level 偏离标准比率超过此值则丢弃）
STD_RATIOS       = (0.0, 0.236, 0.382, 0.5, 0.618, 0.786, 1.0)  # 标准 Fibonacci 比率
TOP_N            = 6                        # 每步长选 top_n 条腿（up/down 各 top_n/2），合并后每步最多 2 组

# ── 趋势腿近期窗口（★ 用滑块交互调参） ──
RECENT_BARS      = 90                       # 基础步长，自动扩展 ×2, ×3（日K: 60-120, 小时K: 168-336）
SKIP_RECENT      = 10                       # 从最新 bar 往前跳过 N 根，不参与分析

# ── 触碰/突破检测（notebook 不直接用，列出供参考） ──
TOUCH_TOLERANCE     = 0.5                   # 触碰检测：price ± tolerance 内视为触碰
TOUCH_COOLDOWN_SEC  = 60.0                  # 同一 fib 线冷却时间（秒）
BREAKOUT_TOLERANCE  = 1.0                   # 突破判定：close 超出 leg 边界 ± tolerance 则失效

In [28]:
# ─── 1. 从缓存读取 klines ───
conn = duckdb.connect(DATA_DB, read_only=True)
_df = conn.execute(
    'SELECT ts, "open", high, low, "close", volume FROM klines '
    'WHERE symbol=? AND "interval"=? ORDER BY ts', [SYMBOL, INTERVAL]).fetchdf()
conn.close()
klines = _df.to_dict("records")
print(f"[{SYMBOL}/{INTERVAL}] klines={len(klines)}")
assert len(klines) > 0, f"No klines found for {SYMBOL}/{INTERVAL} in {DATA_DB}"

# ─── 2. 构建 config 的工厂函数（recent_bars 由滑块动态传入） ───
from dataclasses import dataclass, field

def make_cfg(recent_bars=RECENT_BARS):
    from timing.analysis.algo.retracement.config import RetracementConfig
    return RetracementConfig(
        pivot_windows=PIVOT_WINDOWS, zigzag_thresholds=ZIGZAG_THRESHOLDS,
        regression_windows=REGRESSION_WINDOWS, weights=WEIGHTS,
        cluster_tolerance_pct=CLUSTER_TOLERANCE_PCT, min_cluster_conf=MIN_CLUSTER_CONF,
        min_leg_span_pct=MIN_LEG_SPAN_PCT, max_ratio_error=MAX_RATIO_ERROR,
        std_ratios=STD_RATIOS, top_n=TOP_N, recent_bars=recent_bars,
        skip_recent=SKIP_RECENT,
        touch_tolerance=TOUCH_TOLERANCE, touch_cooldown_sec=TOUCH_COOLDOWN_SEC,
        breakout_tolerance=BREAKOUT_TOLERANCE,
    )

[515300.OF/1d] klines=1608


In [29]:
# ─── 3. 交互式调参：拖动 recent_bars 滑块，实时看多步长 Fib 线变化 ───
# 每步长：top_n/2 up + top_n/2 down → 加权合并 → 每步最多 2 组（1 up + 1 down）
# 步长 ×1, ×2, ×3，共计最多 6 组 FibGroup
STEP_COLORS = {1: "#e74c3c", 2: "#2ecc71", 3: "#3498db"}
STEP_LABELS = {1: "短期 ×1", 2: "中期 ×2", 3: "长期 ×3"}
FIB_COLORS = ["#e377c2", "#ff7f0e", "#2ca02c", "#9467bd", "#8c564b", "#17becf", "#bcbd22"]
out_area = widgets.Output()

def run_and_plot(recent_bars):
    with out_area:
        clear_output(wait=True)
        cfg = make_cfg(recent_bars=recent_bars)
        result = compute_retracement(klines, cfg)
        fdf = result["feature_df"].copy()
        fdf["dt"] = pd.to_datetime(fdf["ts"], unit="ms")
        steps = result["steps"]
        n_total = len(fdf)
        eff_end = result.get("effective_end", n_total)
        print(f"recent_bars={recent_bars}  skip_recent={cfg.skip_recent}  total={n_total}  effective_end={eff_end}")
        for s in steps:
            print(f"  {STEP_LABELS[s['multiplier']]}: target={s['target_bars']} actual={s['actual_bars']} "
                  f"legs={s['raw_legs']} ranked={s['ranked_legs']} up→{s['up_merged']} down→{s['down_merged']} groups={len(s['groups'])}")

        # ═══ 汇总图：全部步长的 Fib 线叠加 ═══
        fig_all = go.Figure()
        fig_all.add_trace(go.Scatter(x=fdf["dt"], y=fdf["close"], mode="lines", name="close", line=dict(color="black", width=1)))
        if eff_end < n_total:
            fig_all.add_vrect(x0=fdf["dt"].iloc[eff_end], x1=fdf["dt"].iloc[-1],
                              fillcolor="gray", opacity=0.10, line_width=0,
                              annotation_text=f"skip {cfg.skip_recent}", annotation_position="top right")
        for step in steps:
            mult = step["multiplier"]
            sc = STEP_COLORS[mult]
            for gi, g in enumerate(step["groups"]):
                leg = g.leg
                sd, ed = pd.to_datetime(leg.start_ts, unit="ms"), pd.to_datetime(leg.end_ts, unit="ms")
                y0 = leg.low if leg.direction == "up" else leg.high
                y1 = leg.high if leg.direction == "up" else leg.low
                fig_all.add_trace(go.Scatter(x=[sd, ed], y=[y0, y1], mode="lines+markers",
                    name=f"{STEP_LABELS[mult]} {g.direction}", line=dict(color=sc, width=2, dash="dash"),
                    marker=dict(size=8, symbol="diamond"), legendgroup=f"s{mult}"))
                for ri, (ratio, price) in enumerate(g.levels):
                    fig_all.add_shape(type="line", x0=sd, x1=fdf["dt"].iloc[-1], y0=price, y1=price,
                                      line=dict(color=sc, width=1, dash="dot"))
                    if ri in (0, 3, 6):
                        fig_all.add_annotation(x=fdf["dt"].iloc[-1], y=price, showarrow=False, xanchor="left",
                            text=f"×{mult}{g.direction[0]} {ratio:.3f} {price:.2f}", font=dict(size=8, color=sc))
        fig_all.update_layout(title=f"{SYMBOL} — 汇总 Fib (base={recent_bars}, skip={cfg.skip_recent}, ×1/×2/×3)",
                              xaxis_title="time", yaxis_title="price", template="plotly_white",
                              xaxis_rangeslider_visible=False,
                              legend=dict(orientation="h", yanchor="bottom", y=1.02), height=600)
        fig_all.show()

        # ═══ 各步长单独一张图：全量 K 线 + 高亮活跃区间 + skip 灰区 + Fib 线 ═══
        for step in steps:
            mult = step["multiplier"]
            sc = STEP_COLORS[mult]
            groups = step["groups"]
            if not groups: continue
            actual_start = step["actual_start"]
            step_eff_end = step.get("effective_end", eff_end)
            fig = go.Figure()
            fig.add_trace(go.Scatter(x=fdf["dt"], y=fdf["close"], mode="lines", name="close", line=dict(color="black", width=1)))
            if eff_end < n_total:
                fig.add_vrect(x0=fdf["dt"].iloc[eff_end], x1=fdf["dt"].iloc[-1],
                              fillcolor="gray", opacity=0.10, line_width=0,
                              annotation_text=f"skip {cfg.skip_recent}", annotation_position="top right")
            if actual_start < step_eff_end:
                fig.add_vrect(x0=fdf["dt"].iloc[actual_start], x1=fdf["dt"].iloc[min(step_eff_end, n_total) - 1],
                              fillcolor=sc, opacity=0.07, line_width=0,
                              annotation_text=f"{STEP_LABELS[mult]} ({step['actual_bars']} bars)",
                              annotation_position="top left")
            for gi, g in enumerate(groups):
                leg = g.leg
                sd, ed = pd.to_datetime(leg.start_ts, unit="ms"), pd.to_datetime(leg.end_ts, unit="ms")
                y0 = leg.low if leg.direction == "up" else leg.high
                y1 = leg.high if leg.direction == "up" else leg.low
                fig.add_trace(go.Scatter(x=[sd, ed], y=[y0, y1], mode="lines+markers",
                    name=f"{g.direction} span={leg.span_pct:.1%}", line=dict(color=sc, width=2, dash="dash"),
                    marker=dict(size=10, symbol="diamond")))
                for ri, (ratio, price) in enumerate(g.levels):
                    fc = FIB_COLORS[ri % len(FIB_COLORS)]
                    fig.add_hline(y=price, line_width=1, line_dash="dot", line_color=fc,
                                  annotation_text=f"{ratio:.3f}  {price:.2f}", annotation_position="right")
                print(f"  {STEP_LABELS[mult]} {g.direction}: low={leg.low:.2f} high={leg.high:.2f} span={leg.span_pct:.2%}")
                print(f"    levels: {[(round(r,3), round(p,2)) for r,p in g.levels]}")
            fig.update_layout(title=f"{SYMBOL} — {STEP_LABELS[mult]} (target={step['target_bars']}, actual={step['actual_bars']})",
                              xaxis_title="time", yaxis_title="price", template="plotly_white",
                              legend=dict(orientation="h"), height=500)
            fig.show()

slider = widgets.IntSlider(value=RECENT_BARS, min=30, max=min(len(klines), 500), step=10, description="recent_bars",
                           style={"description_width": "100px"}, layout=widgets.Layout(width="600px"),
                           continuous_update=False)
slider.observe(lambda change: run_and_plot(change["new"]), names="value")
display(slider, out_area)
run_and_plot(RECENT_BARS)

IntSlider(value=90, continuous_update=False, description='recent_bars', layout=Layout(width='600px'), max=500,…

Output()

# 缓存验证

In [ ]:
# ─── 6. 从缓存加载全量数据 ───
import json, math, asyncio, aiosqlite
import duckdb, pandas as pd
from timing.analysis.algo.retracement.models import TrendLeg, FibGroup

DATA_DB = str(root / "timing" / "cache" / "data.duckdb")
RET_DB  = str(root / "timing" / "cache" / "retracement.sqlite")

# 加载 klines
conn = duckdb.connect(DATA_DB, read_only=True)
klines_map = {}
pairs = conn.execute('SELECT DISTINCT symbol, "interval" FROM klines ORDER BY symbol').fetchall()
for sym, intv in pairs:
    df = conn.execute('SELECT ts, "open", high, low, "close", volume FROM klines WHERE symbol=? AND "interval"=? ORDER BY ts', [sym, intv]).fetchdf()
    df["dt"] = pd.to_datetime(df["ts"], unit="ms")
    klines_map[f"{sym}:{intv}"] = df
conn.close()
print(f"[DataEngine] loaded {len(klines_map)} pairs: {list(klines_map.keys())}")

# 加载 retracement 缓存
def _deserialize(data: dict) -> dict:
    feature_df = pd.DataFrame(data.get("features", []))
    clusters_high_df = pd.DataFrame(data.get("clusters_high", []))
    clusters_low_df = pd.DataFrame(data.get("clusters_low", []))
    groups = []
    for gd in data.get("groups", []):
        leg = TrendLeg(**gd["leg"])
        groups.append(FibGroup(leg=leg, levels=[(lv[0], lv[1]) for lv in gd["levels"]], score=gd["score"], direction=gd["direction"]))
    return {"feature_df": feature_df, "clusters_high_df": clusters_high_df, "clusters_low_df": clusters_low_df,
            "wmap": data.get("wmap", {}), "groups": groups,
            "legs_found": data.get("legs_found", 0), "legs_kept": data.get("legs_kept", 0)}

async def _load_retracement():
    cache = {}
    async with aiosqlite.connect(RET_DB) as db:
        async with db.execute("SELECT key, value FROM kv WHERE key LIKE 'retracement:%'") as cur:
            async for key, val in cur:
                cache[key] = _deserialize(json.loads(val))
    return cache
ret_cache = await _load_retracement()
for k, v in ret_cache.items():
    print(f"  {k}  groups={len(v['groups'])} legs_found={v['legs_found']}")
print(f"[RetracementService] loaded {len(ret_cache)} entries")

In [ ]:
# ─── 7. 批量绘图：每个 symbol 一张总览图（K线 + 全部 Fib 回撤线） ───
import plotly.graph_objects as go

FIB_COLORS = ["#e377c2", "#ff7f0e", "#2ca02c", "#9467bd", "#8c564b", "#17becf", "#bcbd22"]
GROUP_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]

for cache_key, result in ret_cache.items():
    parts = cache_key.split(":")
    symbol, interval = parts[1], parts[2]
    kline_key = f"{symbol}:{interval}"
    kdf = klines_map.get(kline_key)
    if kdf is None or kdf.empty:
        print(f"SKIP {symbol}/{interval}: no klines")
        continue
    groups = result.get("groups", [])
    if not groups:
        print(f"SKIP {symbol}/{interval}: no fib groups")
        continue

    fig = go.Figure()
    fig.add_trace(go.Candlestick(x=kdf["dt"], open=kdf["open"], high=kdf["high"], low=kdf["low"], close=kdf["close"],
                                  name="K-line", increasing_line_color="#d62728", decreasing_line_color="#1f77b4"))

    for gi, g in enumerate(groups):
        leg = g.leg
        gc = GROUP_COLORS[gi % len(GROUP_COLORS)]
        leg_start_dt = pd.to_datetime(leg.start_ts, unit="ms")
        leg_end_dt = pd.to_datetime(leg.end_ts, unit="ms")
        y0 = leg.low if leg.direction == "up" else leg.high
        y1 = leg.high if leg.direction == "up" else leg.low
        fig.add_trace(go.Scatter(x=[leg_start_dt, leg_end_dt], y=[y0, y1], mode="lines+markers",
                                  name=f"G{gi} {g.direction} {leg.span_pct:.1%}", line=dict(color=gc, width=2, dash="dash"),
                                  marker=dict(size=8, symbol="diamond"), legendgroup=f"g{gi}"))
        for ri, (ratio, price) in enumerate(g.levels):
            fc = FIB_COLORS[ri % len(FIB_COLORS)]
            fig.add_shape(type="line", x0=leg_start_dt, x1=kdf["dt"].iloc[-1], y0=price, y1=price,
                          line=dict(color=fc, width=1, dash="dot"))
            fig.add_annotation(x=kdf["dt"].iloc[-1], y=price, text=f"G{gi} {ratio:.3f} {price:.2f}",
                               showarrow=False, xanchor="left", font=dict(size=9, color=fc))

    fig.update_layout(
        title=f"{symbol} / {interval}  —  {len(groups)} Fib Groups  ({len(kdf)} bars)",
        xaxis_title="time", yaxis_title="price", template="plotly_white",
        xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
        height=600,
    )
    fig.show()
    for gi, g in enumerate(groups):
        print(f"  G{gi} [{g.direction}] span={g.leg.span_pct:.2%} score={g.score:.3f}  levels={[(round(r,3), round(p,2)) for r,p in g.levels]}")
    print()